# FinDisputeEval — CFPB Synthetic Seed Selection **v3**（Colab / 本機雙模式）

**v2 執行 + 雙方交叉分析後的整合版**（v1/v2 保留為 baseline；輸出走獨立目錄 `dataset/curated/seed_pools/cfpb_dispute/seed_v03/`）。

## v3 = v2 的九項修正 + 以下四個補丁（v2 run 實測後追加）

| # | v2 run 發現 | v3 修正 |
|---|---|---|
| 1 | **population proxy 被去重優先序污染**：提到 Zelle 的列全被歸給 `zelle_fulltext` 來源，按來源篩選使 Zelle 率 = 0.0、prepaid 格只剩 4 筆 | 改**條件式重建**（Product × 日期，與 source_query 無關） |
| 2 | **特徵池保護性抽樣的第二層偏差**：40k 池把 Zelle/prepaid 全保、其餘只抽 ~1/5——只修 #1 會得到 ~20%+ 的錯誤 Zelle 率 | Step 12 記 `sampling_weight`（保護列=1、抽樣列=抽樣率倒數）；Step 17 population 塊全改**加權統計**（加權比率/分位數/格子分布） |
| 3 | manifest `download_mode` 快取命中時漏記（空字典） | skip 分支記 `cache_hit` |
| 4 | bulk 檔滯後：DATE_MAX 是 2026-07-02 但實際資料只到 2026-06-12 | manifest 記 `effective_data_range`（實際資料起迄） |

## 驗收清單（重跑後檢查）

- [ ] `population_proxy` 的 `zelle_mention` ≈ **0.13**（分母是 claim≠other 的 qpool；全列分母的自然率是 8.31%，兩者都對、分母不同）
- [ ] `population_proxy` 的 `cell_distribution_weighted` 出現 prepaid 三格（merchant ~1,100 / fees ~670 / unauthorized ~390；fraud_scam 在 2025+ 窗口本來就是 0）
- [ ] `estimated_population_rows` ≈ 12 萬
- [ ] `pull_weighted` 的 zelle rate 維持 ~0.216（定義即呈現 pull 設計，不加權）
- [ ] manifest 有 `download_mode: cache_hit` 與 `effective_data_range: … .. 2026-06-12`
- [ ] **再現性檢查**：同 SEED 下 `cfpb_seed_pool.jsonl` 應與 v2 run **byte-identical**（v2 sha256 = `05134011656a…`）

> 沿用紅線：含 raw narrative 的輸出只寫 Drive、不進 repo；`claim_type` 是 candidate 非 gold；272 筆 fraud-scam `relabel_required` 是 M9 golden labels 的原料。

## Step 1 — Environment / Colab bootstrap（dual-path + Drive mount）

In [ ]:
import sys
from pathlib import Path
from datetime import datetime


def progress(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the FinDisputeEval project root.")


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

RAW_DIR = PROJECT_ROOT / "dataset" / "external" / "cfpb" / "raw" / "seed_build_v03_cache_2026-07-04"
OUT_DIR = PROJECT_ROOT / "dataset" / "curated" / "seed_pools" / "cfpb_dispute" / "seed_v03"
MODELS_DIR = PROJECT_ROOT / "dataset" / "cache" / "models"
for directory in (RAW_DIR, OUT_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

progress(f"Step 1/18 complete: IN_COLAB={IN_COLAB}, PROJECT_ROOT={PROJECT_ROOT}")


In [ ]:
progress("Step 2/18: installing/loading dependencies")
# v2 修正 #7：不帶 -U、不裝 requests（Colab 內建 2.32.4，避免 google-colab 依賴衝突）
%pip -q install pydantic pandera rapidfuzz datasketch spacy nltk fasttext-wheel tqdm

import importlib.util, subprocess
if importlib.util.find_spec("requests") is None:      # 只有本機缺才裝
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)

import spacy
try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)

import nltk
nltk.download("vader_lexicon", quiet=True)
progress("Step 2/18 complete: deps ready")

## Step 3 — Provenance 與查詢定義（手寫保留區）

三個 pull：`zelle_fulltext`（缺口補強）、`inscope_recent`（2025+ 四產品）、**`prepaid_alltime`（v2 新增）**——prepaid 2025+ 只有 4,791 筆且原 mapping 全漏接，全期（2011-12 起）實測有 11,269 筆敘事、含可映射的 unauthorized/dispute/fees/fraud 標籤。

In [ ]:
import hashlib, json, platform
from datetime import timezone

SEED = 20260703
API_BASE = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

INSCOPE_DATE_MIN = "2025-01-01"
DATE_MAX         = "2026-07-02"
INSCOPE_PRODUCTS = [
    "Credit card",
    "Checking or savings account",
    "Money transfer, virtual currency, or money service",
    "Prepaid card",
]

PULLS = {
    "zelle_fulltext": {
        "search_term": "Zelle", "field": "complaint_what_happened",
        "date_received_min": "2017-06-01", "date_received_max": DATE_MAX,
    },
    "inscope_recent": {
        "product": INSCOPE_PRODUCTS,
        "date_received_min": INSCOPE_DATE_MIN, "date_received_max": DATE_MAX,
    },
    "prepaid_alltime": {                        # v2 修正 #6：prepaid 全期補強
        "product": ["Prepaid card"],
        "date_received_min": "2011-12-01", "date_received_max": DATE_MAX,
    },
}
EXISTING_EXPORTS = sorted(RAW_DIR.glob("CFPB*2026-06-09*.csv"))

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "notebook": "FinDisputeEval_CFPB_SeedPool_Build_colab_v03",
    "notebook_version": 3,
    "version_changes": ["stratified cap (no gap take-all)", "has_narrative=true", "reference dist split pull_weighted/population_proxy",
                   "literal_eval SyntaxWarning suppressed", "LID: keep all en / exclude non-en prob>=0.5",
                   "prepaid mapping widened + prepaid_alltime pull", "no -U pip / no requests install",
                   "anchor_confidence from candidate supply",
                   "v3: population proxy by criteria (not source_query)",
                   "v3: sampling_weight + weighted population statistics",
                   "v3: cache_hit recorded in download_mode",
                   "v3: effective_data_range in manifest"],
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "api_base": API_BASE,
    "pulls": PULLS,
    "python": platform.python_version(),
    "stages": {}, "inputs": {}, "outputs": {},
}
progress(f"Step 3/18 complete: {len(PULLS)} pulls defined; existing exports: {len(EXISTING_EXPORTS)}")

## Step 4 — 資料下載（API → 年度切塊 → bulk fallback；v2 修正 #2）

v1 run 的教訓：Colab 上 **default UA→403、browser UA→200**（本機相反）；`has_narrative=yes` 無效 → 226-byte 空表頭被當成 IncompleteRead。v2：`has_narrative=true`、headers 順序按環境、空表頭偵測、429 退避 30s 重試一次。

In [ ]:
import requests, time

BULK_URL = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
BULK_CANDIDATES = [Path("/content/complaints.csv.zip"), RAW_DIR / "complaints.csv.zip"]
BULK_LOCAL = BULK_CANDIDATES[0] if IN_COLAB else BULK_CANDIDATES[1]

STD_COLS_BULK = ["Date received", "Product", "Sub-product", "Issue", "Sub-issue",
                 "Consumer complaint narrative", "Company public response", "Company",
                 "State", "ZIP code", "Tags", "Submitted via", "Date sent to company",
                 "Company response to consumer", "Timely response?", "Complaint ID"]

BROWSER_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"),
    "Accept": "text/csv,application/json;q=0.9,*/*;q=0.8",
    "Referer": "https://www.consumerfinance.gov/data-research/consumer-complaints/search/",
}
HEADER_ORDER = (BROWSER_HEADERS, {}) if IN_COLAB else ({}, BROWSER_HEADERS)   # v1 run 實測的環境差異

def _stream_to(r, out_path: Path):
    tmp = out_path.with_suffix(out_path.suffix + ".part")
    with open(tmp, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    tmp.replace(out_path)

class HeaderOnlyCSV(requests.RequestException):
    """HTTP 200 但只回 CSV 表頭（v1 的 has_narrative=yes 症狀）——視為失敗。"""

def download_csv(params: dict, out_path: Path, timeout=1200):
    q = {**params, "format": "csv", "no_aggs": "true", "has_narrative": "true"}   # v2 修正 #2
    last = None
    for headers in HEADER_ORDER:
        for attempt in range(2):
            try:
                with requests.get(API_BASE, params=q, headers=headers, stream=True, timeout=timeout) as r:
                    if r.status_code == 429:
                        progress("[api] 429 rate-limited; backing off 30s")
                        time.sleep(30)
                        continue
                    r.raise_for_status()
                    _stream_to(r, out_path)
                if out_path.stat().st_size < 400:
                    head = out_path.read_text(encoding="utf-8", errors="replace").strip()
                    if head.count("\n") == 0 and head.startswith("Date received"):
                        raise HeaderOnlyCSV(f"header-only CSV ({out_path.stat().st_size} bytes) — 檢查參數")
                return out_path
            except requests.RequestException as e:
                last = e
                code_ = getattr(getattr(e, "response", None), "status_code", None)
                progress(f"[api] headers={'browser' if headers else 'default'} attempt={attempt} -> {code_ or type(e).__name__}")
                if code_ == 403:
                    break                     # 403 換 header profile；同 profile 重試無意義
                time.sleep(3 * (attempt + 1))
    raise last

def ensure_bulk() -> Path:
    for cand in BULK_CANDIDATES:
        if cand.exists() and cand.stat().st_size > 1_000_000_000:
            progress(f"[bulk] 使用既有檔 {cand}")
            return cand
    progress(f"[bulk] 下載 {BULK_URL} …（~1.4GB，一次性）")
    last = None
    for headers in HEADER_ORDER:
        try:
            with requests.get(BULK_URL, headers=headers, stream=True, timeout=3600) as r:
                r.raise_for_status()
                _stream_to(r, BULK_LOCAL)
            progress(f"[bulk] 完成 {BULK_LOCAL.stat().st_size:,} bytes")
            return BULK_LOCAL
        except requests.RequestException as e:
            last = e
            progress(f"[bulk] headers={'browser' if headers else 'default'} -> {getattr(getattr(e, 'response', None), 'status_code', None) or type(e).__name__}")
    raise RuntimeError(
        "bulk 下載也被擋。解法：在本機執行本 cell（本機 IP 可過），"
        "或把本機下載好的 cfpb_*.csv 放到 Drive 的 datasets/cfpb/raw/ 後重跑（會命中快取）。") from last

def bulk_filter(name: str, params: dict) -> list:
    import pandas as pd
    from tqdm.auto import tqdm
    out = RAW_DIR / f"cfpb_{name}.csv"
    bulk = ensure_bulk()
    dmin, dmax = params["date_received_min"], params["date_received_max"]
    products = set(params.get("product", []))
    term = str(params.get("search_term", "")).lower()
    wrote_header, n_out = False, 0
    for chunk in tqdm(pd.read_csv(bulk, dtype=str, keep_default_na=False, chunksize=200_000),
                      desc=f"bulk filter {name}", unit="chunk"):
        chunk.columns = [c.strip() for c in chunk.columns]
        m = chunk["Consumer complaint narrative"].str.len().gt(0)
        d = chunk["Date received"].str.slice(0, 10)
        m &= d.ge(dmin) & d.le(dmax)
        if products:
            m &= chunk["Product"].isin(products)
        if term:
            m &= chunk["Consumer complaint narrative"].str.contains(term, case=False, regex=False)
        sub = chunk.loc[m, STD_COLS_BULK]
        if len(sub):
            sub.to_csv(out, mode="a" if wrote_header else "w", header=not wrote_header, index=False)
            wrote_header = True
            n_out += len(sub)
    if not wrote_header:
        pd.DataFrame(columns=STD_COLS_BULK).to_csv(out, index=False)
    progress(f"[bulk] {name}: 過濾出 {n_out:,} 列 -> {out.name}")
    return [out]

def download_pull(name: str, params: dict) -> list:
    whole = RAW_DIR / f"cfpb_{name}.csv"
    if whole.exists() and whole.stat().st_size > 400:
        progress(f"[skip] {whole.name} 已存在（快取命中）")
        manifest.setdefault("download_mode", {})[name] = "cache_hit"
        return [whole]
    mode = manifest.setdefault("download_mode", {})
    try:
        download_csv(params, whole)
        progress(f"[ok:api] {whole.name} = {whole.stat().st_size:,} bytes")
        mode[name] = "api_whole"
        return [whole]
    except requests.RequestException as e:
        code_ = getattr(getattr(e, "response", None), "status_code", None)
        if code_ == 403 or isinstance(e, HeaderOnlyCSV):
            progress(f"[warn] API 不可用（{code_ or 'header-only'}）→ 官方 bulk fallback")
            mode[name] = "bulk_filter"
            return bulk_filter(name, params)
        progress(f"[warn] 整段失敗（{e}）→ 年度切塊…")
    try:
        files = []
        y0 = int(params["date_received_min"][:4]); y1 = int(params["date_received_max"][:4])
        for y in range(y0, y1 + 1):
            p = dict(params,
                     date_received_min=max(params["date_received_min"], f"{y}-01-01"),
                     date_received_max=min(params["date_received_max"], f"{y}-12-31"))
            fp = RAW_DIR / f"cfpb_{name}_{y}.csv"
            if not (fp.exists() and fp.stat().st_size > 400):
                download_csv(p, fp); time.sleep(2)
            files.append(fp)
        mode[name] = "api_yearly"
        return files
    except requests.RequestException as e:
        progress(f"[warn] 年度切塊也失敗（{e}）→ 官方 bulk fallback")
    mode[name] = "bulk_filter"
    return bulk_filter(name, params)

progress("Step 4/18: downloading CFPB pulls")
pull_files = {name: download_pull(name, params) for name, params in PULLS.items()}
for name, fps in pull_files.items():
    for fp in fps:
        manifest["inputs"][fp.name] = {"sha256": sha256_file(fp), "query": name}
if BULK_LOCAL.exists():
    manifest["inputs"]["complaints.csv.zip"] = {"sha256": sha256_file(BULK_LOCAL), "query": "bulk_fallback_source"}
for fp in EXISTING_EXPORTS:
    manifest["inputs"][fp.name] = {"sha256": sha256_file(fp), "query": "existing_export_2026-06-09"}
progress(f"Step 4/18 complete: {len(manifest['inputs'])} input files; modes={manifest.get('download_mode', {})}")

## Step 5–6 — 載入合併（Complaint ID 去重）+ Pandera 驗證

In [ ]:
import pandas as pd

STD_COLS = STD_COLS_BULK

def load_one(fp: Path, query_name: str) -> pd.DataFrame:
    df = pd.read_csv(fp, dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    missing = [c for c in STD_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{fp.name} 缺欄位: {missing}")
    df = df[STD_COLS].copy()
    df = df[df["Consumer complaint narrative"].str.len() > 0]     # 防禦：空敘事列直接丟
    df["source_query"] = query_name
    return df

progress("Step 5/18: loading and merging CFPB CSV inputs")
frames = []
for name, fps in pull_files.items():
    frames += [load_one(fp, name) for fp in fps]
frames += [load_one(fp, "existing_export") for fp in EXISTING_EXPORTS]
raw = pd.concat(frames, ignore_index=True)

# 去重優先序：zelle > prepaid_alltime > inscope（來源標籤溯源清楚）
prio = {"zelle_fulltext": 0, "prepaid_alltime": 1, "inscope_recent": 2}
raw["__prio"] = raw["source_query"].map(prio).fillna(3)
raw = (raw.sort_values("__prio").drop_duplicates("Complaint ID", keep="first")
          .drop(columns="__prio").reset_index(drop=True))
manifest["stages"]["loaded_unique_complaints"] = int(len(raw))
d = raw["Date received"].str.slice(0, 10)
manifest["stages"]["effective_data_range"] = f"{d.min()} .. {d.max()}"   # bulk 檔滯後於 DATE_MAX，記實際迄日
print(raw["source_query"].value_counts().to_string())
progress(f"Step 5/18 complete: {len(raw):,} unique complaints")

In [ ]:
progress("Step 6/18: validating raw schema with Pandera")
try:
    import pandera.pandas as pa
except ImportError:
    import pandera as pa

raw_schema = pa.DataFrameSchema(
    {
        "Complaint ID": pa.Column(str, pa.Check.str_length(min_value=1), unique=True),
        "Consumer complaint narrative": pa.Column(str, pa.Check.str_length(min_value=1)),
        "Product": pa.Column(str, pa.Check.str_length(min_value=1)),
        "Sub-product": pa.Column(str),
        "Issue": pa.Column(str),
        "Sub-issue": pa.Column(str),
        "Date received": pa.Column(str, pa.Check.str_matches(r"^\d{4}-\d{2}-\d{2}")),
    },
    strict=False, coerce=True,
)
raw = raw_schema.validate(raw, lazy=False)
progress("Step 6/18 complete: Pandera schema PASS")

## Step 7 — 清洗（redaction normalization 手寫保留；v2 修正 #4：抑制 literal_eval 的 SyntaxWarning）

v1 的洗版警告來源：`ast.literal_eval` 解析 `b'...\XXXX...'` 字面值時，Python parser 對字面值**內容**裡的無效跳脫序列發 SyntaxWarning——不是我們的 regex 沒寫 raw string。v2 用 `warnings.catch_warnings` 包住。

In [ ]:
import ast, re, warnings

def unwrap_bytes_literal(t: str) -> str:
    if re.match(r"""^b['"]""", t):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", SyntaxWarning)    # v2 修正 #4
                v = ast.literal_eval(t)
            if isinstance(v, bytes):
                return v.decode("utf-8", errors="replace")
            if isinstance(v, str):
                return v
        except (ValueError, SyntaxError):
            pass
    return t

RE_DATE_BROKEN = re.compile(r"XX/XX/year>")
RE_DATE_MASK   = re.compile(r"\bXX/XX/(?:XXXX|\d{2,4})\b")
RE_AMOUNT      = re.compile(r"\{\$([\d.,]+)\}")
RE_REDACT_RUN  = re.compile(r"(?:\bX{2,}\b[ ]*)+")
RE_SPACE_PUNCT = re.compile(r"[ ]+([,.;:!?])")
RE_MULTISPACE  = re.compile(r"[ \t]{2,}")

def normalize_redaction(t: str):
    amounts = RE_AMOUNT.findall(t)
    t = RE_DATE_BROKEN.sub("[DATE]", t)
    t = RE_DATE_MASK.sub("[DATE]", t)
    t = RE_AMOUNT.sub("[AMOUNT]", t)
    t = RE_REDACT_RUN.sub("[REDACTED] ", t)
    t = RE_SPACE_PUNCT.sub(r"\1", t)
    t = RE_MULTISPACE.sub(" ", t)
    return t.strip(), amounts

progress("Step 7/18: normalizing CFPB redactions")
raw["narrative_raw"] = raw["Consumer complaint narrative"].map(unwrap_bytes_literal)
_norm = raw["narrative_raw"].map(normalize_redaction)
raw["narrative_norm"] = _norm.str[0]
raw["amounts"] = _norm.str[1]
raw["n_ws_tokens"]  = raw["narrative_norm"].str.split().str.len().fillna(0).astype(int)
raw["mask_density"] = (raw["narrative_norm"].str.count(r"\[(?:DATE|AMOUNT|REDACTED)\]")
                       / raw["n_ws_tokens"].clip(lower=1))
manifest["stages"]["bytes_literal_unwrapped"] = int((raw["narrative_raw"] != raw["Consumer complaint narrative"]).sum())
progress(f"Step 7/18 complete: b'' unwrapped={manifest['stages']['bytes_literal_unwrapped']:,}; "
         f"mask_density median={raw['mask_density'].median():.3f}")

## Step 8 — 重複偵測（exact family + MinHashLSH，rapidfuzz 抽驗）

In [ ]:
from datasketch import MinHash, MinHashLSH
from rapidfuzz import fuzz
from tqdm.auto import tqdm
import numpy as np

progress("Step 8/18: template-family deduplication")

def family_signature(t: str) -> str:
    s = re.sub(r"\[(?:DATE|AMOUNT|REDACTED)\]", " ", t.lower())
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()[:400]
    return hashlib.sha1(s.encode()).hexdigest()

def shingles(t: str, k=5):
    toks = re.sub(r"[^a-z0-9 ]+", " ", t.lower()).split()
    return {" ".join(toks[i:i + k]) for i in range(max(1, len(toks) - k + 1))}

raw["family_sig"] = raw["narrative_norm"].map(family_signature)
reps = raw.drop_duplicates("family_sig").copy()
lsh = MinHashLSH(threshold=0.85, num_perm=64)
mh_cache = {}
for idx, txt in tqdm(zip(reps.index, reps["narrative_norm"]), total=len(reps), desc="build MinHash LSH"):
    m = MinHash(num_perm=64)
    for sh in shingles(txt):
        m.update(sh.encode())
    mh_cache[idx] = m
    lsh.insert(str(idx), m)

parent = {}
def find(x):
    while parent.get(x, x) != x:
        parent[x] = parent.get(parent[x], parent[x]); x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

for idx in tqdm(reps.index, desc="query LSH"):
    for other in lsh.query(mh_cache[idx]):
        union(idx, int(other))

reps["template_family"] = [find(i) for i in reps.index]
raw = raw.merge(reps[["family_sig", "template_family"]], on="family_sig", how="left")

fam_sizes = raw["template_family"].value_counts()
multi = fam_sizes[fam_sizes > 1]
if len(multi):
    sample = raw.loc[raw["template_family"] == multi.index[0], "narrative_norm"].head(2).tolist()
    if len(sample) == 2:
        print(f"最大 family 抽驗 token_set_ratio = {fuzz.token_set_ratio(sample[0], sample[1]):.0f}（應 >85）")

raw["is_family_rep"] = ~raw.duplicated("template_family", keep="first")
manifest["stages"]["template_families"] = int(raw["template_family"].nunique())
manifest["stages"]["template_duplicate_rows"] = int((~raw["is_family_rep"]).sum())
progress(f"Step 8/18 complete: families={raw['template_family'].nunique():,}; duplicates={(~raw['is_family_rep']).sum():,}")

## Step 9 — Language ID（v2 修正 #5）

v1 把「`en` 但 prob<0.80」的 2,646 筆誤當非英文排除。v2 政策：**`en` 一律保留**；只有「非 `en` 且 prob≥0.5」才排除進 handoff 探針；其餘（短文本噪音）保留——lower-bound 原則，寧可少殺。

In [ ]:
import fasttext, urllib.request

progress("Step 9/18: language identification with fastText")
lid_bin = MODELS_DIR / "lid.176.bin"
lid_ftz = MODELS_DIR / "lid.176.ftz"
if lid_bin.exists():
    lid_path = lid_bin
else:
    if not lid_ftz.exists():
        urllib.request.urlretrieve(
            "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz", lid_ftz)
    lid_path = lid_ftz

lid = fasttext.load_model(str(lid_path))
probe = raw["narrative_norm"].str.replace("\n", " ", regex=False).str.slice(0, 300).tolist()
labels, probs = lid.predict(probe)
raw["lid_lang"] = [l[0].replace("__label__", "") for l in labels]
raw["lid_prob"] = [float(p[0]) for p in probs]

raw["is_excluded_non_english"] = (raw["lid_lang"] != "en") & (raw["lid_prob"] >= 0.5)   # v2 排除規則
low_conf_en = int(((raw["lid_lang"] == "en") & (raw["lid_prob"] < 0.8)).sum())
manifest["stages"]["low_confidence_english_rows"] = low_conf_en                          # v2 改名
manifest["stages"]["excluded_non_english_rows"] = int(raw["is_excluded_non_english"].sum())
print(raw.loc[raw["is_excluded_non_english"], "lid_lang"].value_counts().head(8).to_string())
progress(f"Step 9/18 complete: excluded non-English={raw['is_excluded_non_english'].sum():,}; "
         f"low-confidence English (kept)={low_conf_en:,}")

## Step 10 — Cell taxonomy 對映（手寫保留區；v2 修正 #6：prepaid 加寬）

Prepaid 依全期實測標籤映射：`Charged for a purchase or transfer you did not make`（745）＋ legacy `Unauthorized transactions/trans. issues`（543）→ unauthorized；`isn't resolving a dispute`（1,846）＋ `Overcharged`（45）→ merchant dispute；`Unexpected or other fees`（1,264）＋ `Fees`（77）→ fees；`Fraud or scam`（310）→ fraud-scam（relabel_required，與 p2p 同規則）。`Trouble using the card` 屬 access 問題，維持 other（進 OOS 池）。

In [ ]:
P2P_SUBPRODUCTS = {"Domestic (US) money transfer", "Mobile or digital wallet"}

def map_cell(product, sub_product, issue, sub_issue):
    """→ (card_type, claim_type, route_hint, in_scope, relabel_required)"""
    il, sl = issue.lower(), sub_issue.lower()
    if product == "Credit card":
        ct, rh = "credit_reg_z", "REG_Z"
        if sub_issue == "Card was charged for something you did not purchase with the card":
            return ct, "unauthorized_charge", rh, True, False
        if sub_issue == "Card opened without my consent or knowledge":
            return ct, "identity_fraud_new_account", rh, True, False
        if sub_issue == "Credit card company isn't resolving a dispute about a purchase on your statement":
            return ct, "merchant_dispute_billing_error", rh, True, False
        if issue == "Fees or interest":
            return ct, "fees_interest", rh, True, False
        return ct, "other", rh, True, False
    if product == "Checking or savings account":
        ct, rh = "debit_reg_e", "REG_E"
        if sub_issue == "Transaction was not authorized":
            return ct, "unauthorized_charge", rh, True, False
        if sub_issue == "Account opened without my consent or knowledge":
            return ct, "identity_fraud_new_account", rh, True, False
        if sub_issue == "Problem using a debit or ATM card":
            return ct, "debit_card_problem_mixed", rh, True, False
        if sub_issue in {"Deposits and withdrawals", "Funds not handled or disbursed as instructed", "Banking errors"}:
            return ct, "transaction_banking_error", rh, True, False
        return ct, "other", rh, True, False
    if product == "Prepaid card":                                   # v2 修正 #6
        ct, rh = "prepaid_reg_e", "REG_E"
        if "you did not make" in sl or "unauthori" in il or "lost or stolen" in il or "loss or theft" in il:
            return ct, "unauthorized_charge", rh, True, False
        if issue == "Fraud or scam":
            return ct, "fraud_scam_authorization_ambiguous", "NEEDS_RELABEL", True, True
        if "isn't resolving a dispute" in sl or "overcharged" in sl:
            return ct, "merchant_dispute_billing_error", rh, True, False
        if "fee" in il:
            return ct, "fees_interest", rh, True, False
        return ct, "other", rh, True, False
    if product == "Money transfer, virtual currency, or money service" and sub_product in P2P_SUBPRODUCTS:
        ct = "p2p_reg_e"
        if issue == "Fraud or scam":
            return ct, "fraud_scam_authorization_ambiguous", "NEEDS_RELABEL", True, True
        if "unauthorized" in il:
            return ct, "unauthorized_charge", "REG_E", True, False
        if issue in {"Other transaction problem", "Wrong amount charged or received", "Money was not available when promised"}:
            return ct, "transaction_banking_error", "REG_E", True, False
        return ct, "other", "REG_E", True, False
    return None, None, None, False, False

progress("Step 10/18: mapping CFPB metadata to FinDisputeEval cells")
mapped = raw.apply(lambda r: map_cell(r["Product"], r["Sub-product"], r["Issue"], r["Sub-issue"]),
                   axis=1, result_type="expand")
mapped.columns = ["card_type", "claim_type", "route_hint", "in_scope", "relabel_required"]
raw = pd.concat([raw, mapped], axis=1)
manifest["stages"]["in_scope_rows"] = int(raw["in_scope"].sum())
print(pd.crosstab(raw.loc[raw.in_scope, "claim_type"], raw.loc[raw.in_scope, "card_type"]).to_string())
progress(f"Step 10/18 complete: in-scope {raw['in_scope'].sum():,} / {len(raw):,}")

## Step 11 — spaCy pipeline 與詞典 matchers（佔位符 retokenize 合併）

In [ ]:
import spacy
from spacy.language import Language
from spacy.matcher import Matcher, PhraseMatcher
from nltk.sentiment import SentimentIntensityAnalyzer

progress("Step 11/18: setting up spaCy matchers and sentiment analyzer")
nlp = spacy.load("en_core_web_sm", disable=["ner"])

PLACEHOLDER_MATCHER = Matcher(nlp.vocab)
for _name in ("DATE", "AMOUNT", "REDACTED"):
    PLACEHOLDER_MATCHER.add(f"PH_{_name}", [[{"ORTH": "["}, {"ORTH": _name}, {"ORTH": "]"}]])

@Language.component("merge_placeholders")
def merge_placeholders(doc):
    spans = spacy.util.filter_spans([doc[s:e] for _, s, e in PLACEHOLDER_MATCHER(doc)])
    with doc.retokenize() as retok:
        for sp in spans:
            retok.merge(sp)
    return doc

if "merge_placeholders" not in nlp.pipe_names:
    nlp.add_pipe("merge_placeholders", first=True)

ESCALATION_TERMS = ["attorney", "lawyer", "sue", "lawsuit", "legal action", "small claims",
                    "cfpb", "ftc", "police report", "better business bureau", "attorney general",
                    "speak to a representative", "talk to a human", "supervisor", "regulator"]
HEDGING_TERMS = ["maybe", "might", "i think", "i believe", "not sure", "pretty sure",
                 "possibly", "perhaps", "i guess", "kind of", "sort of", "seems", "apparently"]
SPEECH_ACT_LEMMAS = ["dispute", "report", "demand", "request", "question", "complain", "refuse", "insist"]

pm_esc = PhraseMatcher(nlp.vocab, attr="LOWER")
pm_esc.add("ESC", [nlp.make_doc(t) for t in ESCALATION_TERMS])
pm_hedge = PhraseMatcher(nlp.vocab, attr="LOWER")
pm_hedge.add("HEDGE", [nlp.make_doc(t) for t in HEDGING_TERMS])
pm_zelle = PhraseMatcher(nlp.vocab, attr="LOWER")
pm_zelle.add("ZELLE", [nlp.make_doc("zelle")])
m_speech = Matcher(nlp.vocab)
m_speech.add("SPEECH_ACT", [[{"LEMMA": {"IN": SPEECH_ACT_LEMMAS}, "POS": "VERB"}]])

sia = SentimentIntensityAnalyzer()
progress(f"Step 11/18 complete: pipeline={nlp.pipe_names}")

## Step 12 — 特徵池（便宜前置閘門 + 40k 護欄；prepaid/Zelle 保留優先）

In [ ]:
progress("Step 12/18: building spaCy feature pool")
MAX_FEATURE_POOL = 40000
PROTECT_CARDS = {"prepaid_reg_e"}          # v2：p2p 已非缺口，僅 prepaid 仍受保護

gate = (~raw["is_excluded_non_english"] & raw["is_family_rep"]
        & raw["mask_density"].le(0.25)
        & raw["n_ws_tokens"].between(60, 800))
pool = raw[gate & raw["in_scope"]].copy()

# v2.2：記錄抽樣權重——保護列（prepaid/Zelle）全保=權重1；其餘按抽樣率倒數加權，
# 讓 Step 17 的 population proxy 能還原母體（否則 Zelle 率被保護性抽樣放大 ~5x）。
pool["sampling_weight"] = 1.0
if len(pool) > MAX_FEATURE_POOL:
    keep = pool[pool["card_type"].isin(PROTECT_CARDS)
                | pool["narrative_norm"].str.contains("zelle", case=False)].copy()
    rest_all = pool.drop(keep.index)
    n_take = max(0, MAX_FEATURE_POOL - len(keep))
    rest = rest_all.sample(n=n_take, random_state=SEED).copy()
    rest["sampling_weight"] = len(rest_all) / max(1, n_take)
    pool = pd.concat([keep, rest])
pool = pool.reset_index(drop=True)
manifest["stages"]["feature_pool_rows"] = int(len(pool))
progress(f"Step 12/18 complete: feature pool={len(pool):,}")

In [ ]:
from tqdm.auto import tqdm

progress("Step 13/18: extracting linguistic features with spaCy")
feats = []
docs = nlp.pipe(pool["narrative_norm"].tolist(), batch_size=64)
for doc, raw_text in tqdm(zip(docs, pool["narrative_raw"]), total=len(pool), desc="spaCy features"):
    n_tok = len(doc)
    ph = sum(t.text in ("[DATE]", "[AMOUNT]", "[REDACTED]") for t in doc)
    feats.append({
        "n_tokens": n_tok,
        "n_sents": sum(1 for _ in doc.sents),
        "mask_density_spacy": ph / max(1, n_tok),
        "neg_count": sum(t.dep_ == "neg" for t in doc),
        "esc_hits": len(pm_esc(doc)),
        "hedge_hits": len(pm_hedge(doc)),
        "zelle_mention": bool(pm_zelle(doc)),
        "speech_act_verbs": sorted({doc[s:e].root.lemma_.lower() for _, s, e in m_speech(doc)}),
        "has_date_ph": any(t.text == "[DATE]" for t in doc),
        "has_amount_ph": any(t.text == "[AMOUNT]" for t in doc),
        "vader_compound": sia.polarity_scores(raw_text[:2000])["compound"],
    })
pool = pd.concat([pool, pd.DataFrame(feats, index=pool.index)], axis=1)
print(pool[["n_tokens", "n_sents", "neg_count", "esc_hits", "hedge_hits", "vader_compound"]]
      .describe().round(2).to_string())
progress(f"Step 13/18 complete: zelle_mention={int(pool['zelle_mention'].sum()):,}")

## Step 14 — 評分與分層抽選（v2 修正 #1、#8）

**全格統一政策**（取消 gap take-all）：每格 = quality top-`CELL_CAP`(100) ∪ 格內 Zelle top-`ZELLE_BONUS_CAP`(100)，單格上限 200。`anchor_confidence` 改依**候選供給量**（cap 前池子大小）：≥100 high、30–99 medium、<30 low。

In [ ]:
progress("Step 14/18: scoring and selecting seed candidates")
CELL_CAP = 100
ZELLE_BONUS_CAP = 100
MIN_SENTS, MIN_TOK, MAX_TOK = 3, 80, 600

q = pool[
    pool["n_sents"].ge(MIN_SENTS) & pool["n_tokens"].between(MIN_TOK, MAX_TOK)
    & pool["mask_density_spacy"].le(0.25) & pool["claim_type"].ne("other")
].copy()

def length_fit(n):
    return 1.0 - min(abs(n - 300) / 300, 1.0)

q["completeness"] = (q["has_date_ph"].astype(int) + q["has_amount_ph"].astype(int)
                     + q["n_sents"].clip(upper=10) / 10)
q["phenomenon"] = (q["neg_count"].clip(upper=3) / 3 + q["hedge_hits"].clip(upper=3) / 3
                   + q["esc_hits"].clip(upper=2) / 2 + q["speech_act_verbs"].str.len().clip(upper=3) / 3)
q["quality_score"] = (0.35 * q["completeness"] / 3 + 0.30 * q["phenomenon"] / 4
                      + 0.20 * (1 - q["mask_density_spacy"] / 0.25)
                      + 0.15 * q["n_tokens"].map(length_fit)).round(4)

def anchor_conf(candidate_supply: int) -> str:      # v2 修正 #8：以供給量判定
    if candidate_supply >= 100: return "high"
    if candidate_supply >= 30:  return "medium"
    return "low"

selected, supply = [], {}
for (ct, cl), grp in q.groupby(["card_type", "claim_type"]):
    grp = grp.sort_values(["quality_score", "Complaint ID"], ascending=[False, True])
    supply[(ct, cl)] = len(grp)
    take = pd.concat([grp.head(CELL_CAP),
                      grp[grp["zelle_mention"]].head(ZELLE_BONUS_CAP)]).drop_duplicates("Complaint ID")
    take = take.copy()
    take["anchor_confidence"] = anchor_conf(len(grp))
    selected.append(take)
seeds = pd.concat(selected).reset_index(drop=True)

manifest["stages"]["quality_filtered_candidates"] = int(len(q))
manifest["stages"]["seed_rows"] = int(len(seeds))
manifest["cell_candidate_supply"] = {f"{ct}|{cl}": n for (ct, cl), n in supply.items()}
print(seeds.groupby(["card_type", "claim_type"]).agg(
    n=("Complaint ID", "count"), zelle=("zelle_mention", "sum"),
    q50=("quality_score", "median"), conf=("anchor_confidence", "first")).round(3).to_string())
progress(f"Step 14/18 complete: seed pool={len(seeds):,}（候選 {len(q):,}）")

## Step 15 — Pydantic v2 schema 驗證 + 匯出 seed JSONL（enum 待與 annotation_guidelines_v4 對齊）

In [ ]:
from enum import Enum
from typing import List
from pydantic import BaseModel, Field

class CardType(str, Enum):
    credit_reg_z = "credit_reg_z"; debit_reg_e = "debit_reg_e"
    p2p_reg_e = "p2p_reg_e"; prepaid_reg_e = "prepaid_reg_e"

class ClaimType(str, Enum):
    unauthorized_charge = "unauthorized_charge"
    merchant_dispute_billing_error = "merchant_dispute_billing_error"
    fraud_scam_authorization_ambiguous = "fraud_scam_authorization_ambiguous"
    identity_fraud_new_account = "identity_fraud_new_account"
    fees_interest = "fees_interest"
    transaction_banking_error = "transaction_banking_error"
    debit_card_problem_mixed = "debit_card_problem_mixed"

class RouteHint(str, Enum):
    REG_E = "REG_E"; REG_Z = "REG_Z"; NEEDS_RELABEL = "NEEDS_RELABEL"

class AnchorConfidence(str, Enum):
    high = "high"; medium = "medium"; low = "low"

class LinguisticSignals(BaseModel):
    n_tokens: int; n_sents: int
    mask_density: float
    neg_count: int; hedge_hits: int; esc_hits: int
    speech_act_verbs: List[str]
    vader_compound: float

class ScenarioSeedCandidate(BaseModel):
    seed_id: str
    complaint_id: str
    card_type: CardType
    claim_type: ClaimType
    route_hint: RouteHint
    relabel_required: bool
    zelle_mention: bool
    anchor_confidence: AnchorConfidence
    candidate_source: str = "cfpb_metadata_rule"
    source_query: str
    date_received: str
    company: str
    amounts: List[str] = Field(default_factory=list)
    linguistic: LinguisticSignals
    narrative_norm: str
    narrative_raw: str
    quality_score: float

def to_record(r) -> ScenarioSeedCandidate:
    return ScenarioSeedCandidate(
        seed_id=f"cfpb-{r['Complaint ID']}",
        complaint_id=r["Complaint ID"], card_type=r["card_type"], claim_type=r["claim_type"],
        route_hint=r["route_hint"], relabel_required=bool(r["relabel_required"]),
        zelle_mention=bool(r["zelle_mention"]), anchor_confidence=r["anchor_confidence"],
        source_query=r["source_query"], date_received=r["Date received"][:10], company=r["Company"],
        amounts=list(r["amounts"]),
        linguistic=LinguisticSignals(
            n_tokens=int(r["n_tokens"]), n_sents=int(r["n_sents"]),
            mask_density=float(r["mask_density_spacy"]), neg_count=int(r["neg_count"]),
            hedge_hits=int(r["hedge_hits"]), esc_hits=int(r["esc_hits"]),
            speech_act_verbs=list(r["speech_act_verbs"]), vader_compound=float(r["vader_compound"])),
        narrative_norm=r["narrative_norm"], narrative_raw=r["narrative_raw"],
        quality_score=float(r["quality_score"]))

progress("Step 15/18: validating and exporting seed JSONL")
seed_path = OUT_DIR / "cfpb_seed_pool.jsonl"
with open(seed_path, "w", encoding="utf-8") as f:
    for _, r in seeds.iterrows():
        f.write(to_record(r).model_dump_json() + "\n")
schema_path = OUT_DIR / "scenario_seed_candidate.schema.json"
schema_path.write_text(json.dumps(ScenarioSeedCandidate.model_json_schema(), indent=2), encoding="utf-8")
manifest["outputs"][seed_path.name] = {"sha256": sha256_file(seed_path), "rows": int(len(seeds))}
manifest["outputs"][schema_path.name] = {"sha256": sha256_file(schema_path)}
progress(f"Step 15/18 complete: {len(seeds):,} seeds exported（Pydantic 全數驗證通過）")

## Step 16 — OOS 負樣本池 + 非英語 LID 探針

In [ ]:
progress("Step 16/18: exporting OOS negatives and LID handoff probe")
oos = pool[pool["claim_type"].eq("other") & pool["n_sents"].ge(2)]
oos = oos.sample(n=min(1500, len(oos)), random_state=SEED)
oos_path = OUT_DIR / "oos_negative_pool.jsonl"
with open(oos_path, "w", encoding="utf-8") as f:
    for _, r in oos.iterrows():
        f.write(json.dumps({"complaint_id": r["Complaint ID"], "card_type": r["card_type"],
                            "label": "out_of_scope", "narrative_norm": r["narrative_norm"],
                            "source_query": r["source_query"]}, ensure_ascii=False) + "\n")

probe = raw[raw["is_excluded_non_english"] & raw["narrative_norm"].str.len().ge(60)].head(50)
probe_path = OUT_DIR / "lid_handoff_probe.jsonl"
with open(probe_path, "w", encoding="utf-8") as f:
    for _, r in probe.iterrows():
        f.write(json.dumps({"complaint_id": r["Complaint ID"], "lid_lang": r["lid_lang"],
                            "lid_prob": round(float(r["lid_prob"]), 3),
                            "narrative_norm": r["narrative_norm"][:800],
                            "expected_action": "bilingual_human_handoff"}, ensure_ascii=False) + "\n")
for p, df_ in ((oos_path, oos), (probe_path, probe)):
    manifest["outputs"][p.name] = {"sha256": sha256_file(p), "rows": int(len(df_))}
progress(f"Step 16/18 complete: OOS={len(oos):,}; LID probe={len(probe):,}")

## Step 17 — 參考分布（v2 修正 #3：拆 pull_weighted / population_proxy）

`pull_weighted` 含 Zelle 超採與 prepaid 全期補強的設計偏差（只能與**同設計**子集比對）；`population_proxy` 只用 `inscope_recent` 來源（乾淨的產品×日期過濾），對 2025+ 窗口是無偏的母體代理——**§8.13 的 KL/JS 閘門用這一塊**。

In [ ]:
progress("Step 17/18: exporting reference distributions")
import numpy as np

def wquantile(x, w, qs=(5, 25, 50, 75, 95)):
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    idx = np.argsort(x); x, w = x[idx], w[idx]
    cw = (np.cumsum(w) - 0.5 * w) / w.sum()
    return {str(p): float(np.interp(p / 100, cw, x)) for p in qs}

def dist_block(df_):
    """未加權版（pull_weighted 用——本來就要呈現 pull 設計下的分布）。"""
    return {
        "source_rows": int(len(df_)),
        "cell_distribution": {f"{ct}|{cl}": int(n) for (ct, cl), n in
                              df_.groupby(["card_type", "claim_type"]).size().items()},
        "token_length_quantiles": {str(p): float(df_["n_tokens"].quantile(p / 100)) for p in (5, 25, 50, 75, 95)},
        "sentence_count_quantiles": {str(p): float(df_["n_sents"].quantile(p / 100)) for p in (5, 25, 50, 75, 95)},
        "phenomenon_rates": {
            "negation_present": float(df_["neg_count"].gt(0).mean()),
            "hedging_present": float(df_["hedge_hits"].gt(0).mean()),
            "escalation_language_present": float(df_["esc_hits"].gt(0).mean()),
            "zelle_mention": float(df_["zelle_mention"].mean()),
        },
        "vader_compound_quantiles": {str(p): float(df_["vader_compound"].quantile(p / 100)) for p in (5, 25, 50, 75, 95)},
        "speech_act_verb_counts": pd.Series(
            [v for lst in df_["speech_act_verbs"] for v in lst]).value_counts().head(20).to_dict(),
    }

def dist_block_weighted(df_):
    """加權版（population proxy 用）——以 sampling_weight 還原特徵池保護性抽樣前的母體。"""
    w = df_["sampling_weight"].to_numpy(dtype=float)
    sac = {}
    for lst, wt in zip(df_["speech_act_verbs"], w):
        for v in lst:
            sac[v] = sac.get(v, 0.0) + wt
    return {
        "source_rows_unweighted": int(len(df_)),
        "estimated_population_rows": round(float(w.sum())),
        "cell_distribution_weighted": {f"{ct}|{cl}": round(float(g["sampling_weight"].sum()))
                                       for (ct, cl), g in df_.groupby(["card_type", "claim_type"])},
        "token_length_quantiles": wquantile(df_["n_tokens"], w),
        "sentence_count_quantiles": wquantile(df_["n_sents"], w),
        "phenomenon_rates": {
            "negation_present": float(np.average(df_["neg_count"].gt(0), weights=w)),
            "hedging_present": float(np.average(df_["hedge_hits"].gt(0), weights=w)),
            "escalation_language_present": float(np.average(df_["esc_hits"].gt(0), weights=w)),
            "zelle_mention": float(np.average(df_["zelle_mention"], weights=w)),
        },
        "vader_compound_quantiles": wquantile(df_["vader_compound"], w),
        "speech_act_verb_counts_weighted": {k: round(v) for k, v in
                                            sorted(sac.items(), key=lambda kv: -kv[1])[:20]},
    }

qpool = pool[pool["claim_type"].ne("other")]
# v2.1：條件式篩選（產品 x 日期，與來源標籤無關）；v2.2：再以抽樣權重加權還原母體
qpool_pop = qpool[qpool["Date received"].str.slice(0, 10).ge(INSCOPE_DATE_MIN)
                  & qpool["Product"].isin(INSCOPE_PRODUCTS)]

ref = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_version": 3,
    "pull_weighted": {
        **dist_block(qpool),
        "sampling_design_note": ("BIASED BY PULL DESIGN: zelle_fulltext deliberately oversamples Zelle narratives "
                                 "and prepaid_alltime extends prepaid history to 2011. Use only for comparisons "
                                 "against synthetic sets built with the same pull design."),
    },
    "population_proxy_inscope_2025plus": {
        **dist_block_weighted(qpool_pop),
        "note": ("Population proxy for 2025-01+ x four in-scope products. Selected by product x date CRITERIA "
                 "(source-independent, v2.1) and re-weighted by feature-pool inclusion probability to undo the "
                 "zelle/prepaid protection oversampling (v2.2). USE THIS BLOCK for the section-8.13 KL/JS gate. "
                 "Sanity anchors (independent local replication): ~0.064 after exact template-family dedup (this block's universe; LSH merges push it slightly lower), ~0.13 WITHOUT dedup. Zelle narratives are heavily templated: only ~31% survive family dedup vs ~65% for non-Zelle (mass template complaint campaigns) — dedup is the correct reference universe for 8.13."),
    },
    "anchoring_boundary_note": ("CFPB narratives are single-voice written complaints: distributions apply to "
                                "customer-side utterances only; dialogue-structure baselines come from ABCD/talkmap."),
}
ref_path = OUT_DIR / "cfpb_reference_distributions.json"
ref_path.write_text(json.dumps(ref, indent=2, ensure_ascii=False), encoding="utf-8")
manifest["outputs"][ref_path.name] = {"sha256": sha256_file(ref_path)}
print("pull_weighted zelle rate:", round(ref["pull_weighted"]["phenomenon_rates"]["zelle_mention"], 4))
print("population   zelle rate (weighted):",
      round(ref["population_proxy_inscope_2025plus"]["phenomenon_rates"]["zelle_mention"], 4))
progress("Step 17/18 complete")

## Step 18 — Manifest 與輸出摘要

In [ ]:
import pydantic, pandera, datasketch, rapidfuzz, nltk as _nltk, requests as _requests

progress("Step 18/18: writing manifest and output summary")
manifest["versions"] = {
    "pandas": pd.__version__, "spacy": spacy.__version__, "pydantic": pydantic.__version__,
    "pandera": pandera.__version__, "datasketch": datasketch.__version__,
    "rapidfuzz": rapidfuzz.__version__, "nltk": _nltk.__version__, "requests": _requests.__version__,
    "spacy_model": "en_core_web_sm", "lid_model": lid_path.name,
}
man_path = OUT_DIR / "seed_selection_manifest.json"
man_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(manifest["stages"], indent=2))
print("\n輸出（全部在 Drive，含 raw narrative 的檔案不進 repo）:")
for name, meta in manifest["outputs"].items():
    print(f"  {name}  rows={meta.get('rows','-')}  sha256={meta['sha256'][:12]}…")
progress("Step 18/18 complete")

## 誠實聲明與下一步（v2 更新）

1. `claim_type` 仍為 CFPB 官方標籤映射（candidate，非 gold）；p2p 與 prepaid 的 `fraud_scam_authorization_ambiguous` 皆 `relabel_required=True`。
2. `population_proxy` 只對「2025+ 四產品」窗口無偏；跨窗口比較（如 prepaid 全期）仍需注意時代分布差異。
3. `neg_count` 是 dependency candidate 訊號；LID 是 lower-bound 訊號；CFPB 錨定內容與客戶端語言、不錨定對話結構——三條邊界與 v1 相同。
4. v2 預期 seed 池規模：11–13 格 × ≤200 ≈ **1,500–2,200 筆**（v1 是失衡的 17,669）；每格品質中位數應全面高於 v1 的 p2p 格。
5. 下一步不變（計畫書 v10 §7.4）：seed 合流 schema → NeMo 客製 recipe → 50–100 筆試產 → 用 `population_proxy` 跑 KL/JS → 放量。

In [ ]:
import csv, json, random
from pathlib import Path

def _audit_progress(msg: str):
    if "progress" in globals():
        progress(msg)
    else:
        print(msg, flush=True)

project_root = PROJECT_ROOT if "PROJECT_ROOT" in globals() else find_project_root(Path.cwd())
seed_dir = OUT_DIR if "OUT_DIR" in globals() else project_root / "dataset" / "curated" / "seed_pools" / "cfpb_dispute" / "seed_v03"
audit_dir = project_root / "dataset" / "curated" / "annotations" / "cfpb_seed_review" / "round_001"
src = seed_dir / "cfpb_seed_pool.jsonl"
out = audit_dir / "manual_audit_sample.csv"

if not src.exists():
    raise FileNotFoundError(f"Could not find the v3 seed pool: {src}")

_audit_progress(f"Manual audit sample source: {src}")
rows = []
with src.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
_audit_progress(f"Loaded seed candidates: {len(rows):,}")

targets = []
rng = random.Random(20260704)

def take(label, pred, n=25):
    pool = [r for r in rows if pred(r)]
    pool = sorted(pool, key=lambda r: float(r["quality_score"]))
    low_quality = pool[:min(10, len(pool))]
    rest = pool[len(low_quality):]
    sample = low_quality + rng.sample(rest, min(max(0, n - len(low_quality)), len(rest)))
    _audit_progress(f"{label}: pool={len(pool):,}, sampled={len(sample):,}")
    for r in sample:
        targets.append({
            "audit_group": label,
            "seed_id": r["seed_id"],
            "complaint_id": r["complaint_id"],
            "card_type": r["card_type"],
            "claim_type": r["claim_type"],
            "route_hint": r["route_hint"],
            "relabel_required": r["relabel_required"],
            "zelle_mention": r["zelle_mention"],
            "anchor_confidence": r["anchor_confidence"],
            "quality_score": r["quality_score"],
            "date_received": r["date_received"],
            "company": r["company"],
            "n_tokens": r["linguistic"]["n_tokens"],
            "n_sents": r["linguistic"]["n_sents"],
            "vader_compound": r["linguistic"]["vader_compound"],
            "narrative_norm": r["narrative_norm"][:3000],
            "audit_label_ok": "",
            "audit_notes": "",
        })

take("low_q_debit_identity_fraud", lambda r: r["card_type"] == "debit_reg_e" and r["claim_type"] == "identity_fraud_new_account")
take("low_q_prepaid_fraud_scam", lambda r: r["card_type"] == "prepaid_reg_e" and r["claim_type"] == "fraud_scam_authorization_ambiguous")
take("needs_relabel", lambda r: r["route_hint"] == "NEEDS_RELABEL", n=40)

if not targets:
    raise ValueError("No audit targets were selected; check predicates and seed pool.")

out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(targets[0].keys()))
    w.writeheader()
    w.writerows(targets)

print(out)
print(f"manual audit rows: {len(targets):,}")
print("Fill audit_label_ok with yes/no/unsure and audit_notes with the reason.")